# Ch.4 — One-Class SVM

> Draw a tight boundary around normal data in kernel space. Points outside the boundary are anomalies. No fraud labels needed for training.

**Dataset:** Credit Card Fraud Detection — 284,807 transactions, 0.17% fraud  
**Task:** Learn normal boundary with RBF kernel, target ~75% recall @ 0.5% FPR

## The Core Idea

One-Class SVM separates data from the origin in kernel space with maximum margin:

$$\min_{\mathbf{w}, \rho, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + \frac{1}{\nu n}\sum \xi_i - \rho$$

- $\nu$ controls boundary tightness (fraction of allowed outliers)
- RBF kernel maps to infinite-dimensional space: $K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2)$
- Decision: $f(\mathbf{x}) = \text{sign}(\mathbf{w} \cdot \phi(\mathbf{x}) - \rho)$

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
IMG_DIR = "img/"

import os
os.makedirs(IMG_DIR, exist_ok=True)
np.random.seed(SEED)
print("Libraries loaded.")

In [ ]:
# ── Load, split, and subsample ─────────────────────────────────────────────
df = pd.read_csv("creditcard.csv")
if os.environ.get("CI"):  # limit rows for CI runs
    df = df.head(1000)

X = df.drop("Class", axis=1).values
y = df["Class"].values
feature_names = [c for c in df.columns if c != "Class"]

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Normal training data only
X_normal = X_train[y_train == 0]

# CRITICAL: Subsample for OC-SVM (O(n²) memory for kernel matrix)
subsample_size = 10000
sub_idx = np.random.choice(len(X_normal), size=min(subsample_size, len(X_normal)), replace=False)
X_sub = X_normal[sub_idx]

# Scale
scaler = StandardScaler()
X_sub_s = scaler.fit_transform(X_sub)
X_test_s = scaler.transform(X_test)

print(f"Full normal training: {len(X_normal):,}")
print(f"Subsampled for OC-SVM: {len(X_sub):,}")
print(f"Test: {len(X_test):,} ({y_test.sum()} fraud)")

In [ ]:
# ── Train One-Class SVM ────────────────────────────────────────────────────
print("Training One-Class SVM (this may take a minute)...")
start = time.time()

ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01,
)
ocsvm.fit(X_sub_s)

train_time = time.time() - start
print(f"Training time: {train_time:.1f}s")
print(f"Support vectors: {ocsvm.support_vectors_.shape[0]:,} "
      f"({ocsvm.support_vectors_.shape[0]/subsample_size:.1%} of training data)")

In [ ]:
# ── Score and evaluate ─────────────────────────────────────────────────────
raw_scores = ocsvm.decision_function(X_test_s)
scores_svm = -raw_scores  # higher = more anomalous

fpr_svm, tpr_svm, thresh_svm = roc_curve(y_test, scores_svm)
auc_svm = auc(fpr_svm, tpr_svm)

idx_005 = np.where(fpr_svm <= 0.005)[0][-1]
recall_svm = tpr_svm[idx_005]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr_svm, tpr_svm, color="#16a085", linewidth=2, label=f"OC-SVM (AUC={auc_svm:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].scatter([fpr_svm[idx_005]], [tpr_svm[idx_005]], color="#e74c3c", s=100, zorder=5,
               label=f"@ 0.5% FPR: {recall_svm:.1%}")
axes[0].set(title="ROC — One-Class SVM", xlabel="FPR", ylabel="Recall")
axes[0].legend()

fraud_mask = y_test == 1
axes[1].hist(scores_svm[~fraud_mask], bins=100, alpha=0.6, color="#27ae60",
            label="Legitimate", density=True)
axes[1].hist(scores_svm[fraud_mask], bins=50, alpha=0.7, color="#e74c3c",
            label="Fraud", density=True)
axes[1].set(title="Decision Function Score Distribution", xlabel="Score", ylabel="Density")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}roc_ocsvm.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nRecall @ 0.5% FPR: {recall_svm:.2%}")

## Visualizing the Decision Boundary (2D Projection)

Project data to 2D via PCA and visualize the OC-SVM boundary.

In [ ]:
# ── 2D boundary visualization ──────────────────────────────────────────────
pca = PCA(n_components=2, random_state=SEED)
X_2d_train = pca.fit_transform(X_sub_s[:2000])  # use subset for speed

ocsvm_2d = OneClassSVM(kernel="rbf", gamma="scale", nu=0.01)
ocsvm_2d.fit(X_2d_train)

# Create decision boundary grid
xx, yy = np.meshgrid(
    np.linspace(X_2d_train[:, 0].min() - 2, X_2d_train[:, 0].max() + 2, 300),
    np.linspace(X_2d_train[:, 1].min() - 2, X_2d_train[:, 1].max() + 2, 300),
)
Z = ocsvm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Project test data
X_2d_test = pca.transform(X_test_s)

fig, ax = plt.subplots(figsize=(10, 8))
ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap="Blues_r", alpha=0.3)
ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors="#2c3e50")
ax.scatter(X_2d_test[~fraud_mask, 0][:2000], X_2d_test[~fraud_mask, 1][:2000],
           s=5, alpha=0.3, color="#27ae60", label="Legitimate")
ax.scatter(X_2d_test[fraud_mask, 0], X_2d_test[fraud_mask, 1],
           s=30, alpha=0.8, color="#e74c3c", marker="x", label="Fraud")
ax.set(title="One-Class SVM Decision Boundary (2D PCA Projection)",
       xlabel="PC1", ylabel="PC2")
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}decision_boundary_2d.png", dpi=150, bbox_inches="tight")
plt.show()

## ν Parameter Sensitivity

$\nu$ controls the fraction of training points allowed outside the boundary. For clean training data, this acts as a regularization parameter.

In [ ]:
# ── ν parameter sweep ──────────────────────────────────────────────────────
nu_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
results_nu = []

for nu in nu_values:
    model = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    model.fit(X_sub_s)
    scores = -model.decision_function(X_test_s)
    fpr_i, tpr_i, _ = roc_curve(y_test, scores)
    auc_i = auc(fpr_i, tpr_i)
    idx = np.where(fpr_i <= 0.005)[0][-1]
    n_sv = model.support_vectors_.shape[0]
    results_nu.append({"nu": nu, "AUC": auc_i, "Recall@0.5%FPR": tpr_i[idx],
                       "Support Vectors": n_sv})
    print(f"  ν={nu:.3f}: Recall={tpr_i[idx]:.2%}, SVs={n_sv}")

nu_df = pd.DataFrame(results_nu)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogx(nu_df["nu"], nu_df["Recall@0.5%FPR"], "o-", color="#16a085", linewidth=2)
axes[0].axhline(0.80, color="#e74c3c", linestyle="--", alpha=0.5, label="Target: 80%")
axes[0].set(title="ν vs Recall @ 0.5% FPR", xlabel="ν", ylabel="Recall")
axes[0].legend()

axes[1].semilogx(nu_df["nu"], nu_df["Support Vectors"], "o-", color="#2980b9", linewidth=2)
axes[1].set(title="ν vs Number of Support Vectors", xlabel="ν", ylabel="# SVs")

plt.tight_layout()
plt.savefig(f"{IMG_DIR}nu_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Inference latency measurement ──────────────────────────────────────────
# How fast is OC-SVM at scoring individual transactions?
n_trials = 100
latencies = []

for i in range(n_trials):
    x_single = X_test_s[i:i+1]
    start = time.perf_counter()
    _ = ocsvm.decision_function(x_single)
    latencies.append((time.perf_counter() - start) * 1000)

print(f"Single-transaction inference latency:")
print(f"  Mean: {np.mean(latencies):.2f}ms")
print(f"  P95:  {np.percentile(latencies, 95):.2f}ms")
print(f"  P99:  {np.percentile(latencies, 99):.2f}ms")
print(f"  {'✅ Under 100ms' if np.percentile(latencies, 99) < 100 else '❌ Too slow'}")

## Progress Check

| Constraint | Target | Status |
|------------|--------|--------|
| #1 DETECTION | >80% recall | ❌ ~75% — but 4 complementary detectors now! |
| #2 PRECISION | <0.5% FPR | ✅ ROC thresholding |
| #3 REAL-TIME | <100ms | ✅ ~20ms inference |
| #4 ADAPTABILITY | Drift handling | ❌ Slow retraining |
| #5 EXPLAINABILITY | Justifiable | ⚡ Closest support vectors provide some insight |

**Next**: Ch.5 — Ensemble (combine all 4 detectors → 83% recall!)

## Exercises

In [ ]:
# ── Exercise 1: Gamma sweep ────────────────────────────────────────────────
# TODO: Test gamma in [0.001, 0.01, 0.1, 'scale', 1.0, 10.0]
# For each, train OC-SVM on the subsampled data
# Plot recall vs gamma. What happens at extreme values?

pass

In [ ]:
# ── Exercise 2: Feature selection ──────────────────────────────────────────
# TODO: Train OC-SVM on only the top 10 features (by discrimination from Ch.1)
# Compare recall vs full 30-feature model
# Does reducing dimensionality help the kernel method?

pass

In [ ]:
# ── Exercise 3: Subsample size impact ──────────────────────────────────────
# TODO: Test subsample sizes [1000, 5000, 10000, 20000]
# For each, measure both recall AND training time
# Plot the trade-off. Is 10k the sweet spot?

pass

## Additional Visualization — Threshold Operating Curve

Instead of only one operating point (0.5% FPR), visualize recall and precision across score thresholds.

In [ ]:
# ── Threshold operating curve: recall and precision vs threshold ───────────
threshold_grid = np.quantile(scores_svm, np.linspace(0.80, 0.999, 40))
operating_rows = []

for t in threshold_grid:
    pred = (scores_svm >= t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    tn = ((pred == 0) & (y_test == 0)).sum()

    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    operating_rows.append({"threshold": t, "recall": recall, "precision": precision, "fpr": fpr})

op_df = pd.DataFrame(operating_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(op_df["threshold"], op_df["recall"], color="#16a085", label="Recall")
axes[0].plot(op_df["threshold"], op_df["precision"], color="#c0392b", label="Precision")
axes[0].set(title="Threshold vs Recall/Precision", xlabel="Anomaly score threshold")
axes[0].legend()

axes[1].plot(op_df["fpr"], op_df["recall"], color="#2980b9", linewidth=2)
axes[1].scatter([fpr_svm[idx_005]], [recall_svm], color="#e74c3c", s=70, label="0.5% FPR point")
axes[1].set(title="Operating Frontier", xlabel="FPR", ylabel="Recall")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}operating_curve_ocsvm.png", dpi=150, bbox_inches="tight")
plt.show()

## Additional Visualization — Support Region Diagnostics

Check support-vector density and score quantiles to understand boundary tightness.

In [ ]:
# ── Support-vector ratio and score quantiles ────────────────────────────────
sv_ratio = ocsvm.support_vectors_.shape[0] / len(X_sub_s)
print(f"Support-vector ratio: {sv_ratio:.2%}")

q_levels = [0.50, 0.75, 0.90, 0.95, 0.99]
q_legit = np.quantile(scores_svm[y_test == 0], q_levels)
q_fraud = np.quantile(scores_svm[y_test == 1], q_levels)

diag_df = pd.DataFrame({
    "quantile": q_levels,
    "legitimate": q_legit,
    "fraud": q_fraud,
    "gap(fraud-legit)": q_fraud - q_legit,
})
print(diag_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(["Support", "Non-support"], [sv_ratio, 1 - sv_ratio], color=["#8e44ad", "#95a5a6"])
axes[0].set(title="OC-SVM Boundary Tightness", ylabel="Fraction of sampled normal training set")

axes[1].plot(q_levels, q_legit, "o-", label="Legitimate", color="#27ae60")
axes[1].plot(q_levels, q_fraud, "o-", label="Fraud", color="#e74c3c")
axes[1].set(title="OC-SVM Score Quantiles by Class", xlabel="Quantile", ylabel="Anomaly score")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}support_region_diagnostics_ocsvm.png", dpi=150, bbox_inches="tight")
plt.show()